### Random Baseline Distribution

- Time Randomised [5-30] Days

In [1]:
from pathlib import Path
import sys
import pickle
import random

import numpy as np
import pandas as pd


In [2]:

# ============================================================
# 1. Basic setup
# ============================================================

SEED = 21894
random.seed(SEED)
np.random.seed(SEED)
rng = random.Random(SEED)

DEMETER_ROOT = Path(r"C:\PROJECT-DEMETER\DICLUB\GRID-METHODS\LSTM-AE-HO\Demeter")

if str(DEMETER_ROOT) not in sys.path:
    sys.path.insert(0, str(DEMETER_ROOT))

from lstm import HalfOrbitPairDataset
from lstm import SeismicCriteria

DATA_DIR = DEMETER_ROOT / "Data"
BG_WINDOW_DIR = DATA_DIR / "Bg_window_data"
RESULT_DIR = DEMETER_ROOT / "Final" / "Results" / "Random_baseline_simple"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

EQ_PATH = DATA_DIR / "Main_earthquakes.csv"

MIN_DATA_POINTS = 17
NUM_SAMPLES = 200
NUM_ITERATIONS = 1000

SPATIAL_WIDTH = 22
TIME_WINDOWS = [6, 12, 24, 48]   # change 48 to 28 only if you really meant 28 h

RANDOM_OFFSET_MIN_DAYS = 5
RANDOM_OFFSET_MAX_DAYS = 30

TRAIN_MONTHS = 12
VAL_MONTHS = 3
STRIDE_MONTHS = 3

WINDOW_START_DATE = "2005-01-01 00:00:00"
WINDOW_END_DATE = "2011-01-02 00:00:00"

print("Result folder:", RESULT_DIR)



Result folder: C:\PROJECT-DEMETER\DICLUB\GRID-METHODS\LSTM-AE-HO\Demeter\Final\Results\Random_baseline_simple


In [3]:

# ============================================================
# 2. Load earthquake catalogue
# ============================================================

eq = pd.read_csv(EQ_PATH, parse_dates=["Time"])
eq["Time"] = pd.to_datetime(eq["Time"], errors="coerce").dt.tz_localize(None)
eq = eq.dropna(subset=["Time"]).copy()

print("Earthquake catalogue loaded:", eq.shape)



Earthquake catalogue loaded: (11300, 6)


In [4]:

# ============================================================
# 3. Generate rolling-window dates
# ============================================================

windows = []

current = pd.to_datetime(WINDOW_START_DATE)
final = pd.to_datetime(WINDOW_END_DATE)

while current + pd.DateOffset(months=TRAIN_MONTHS + VAL_MONTHS) < final:
    train_start = current
    train_end = current + pd.DateOffset(months=TRAIN_MONTHS)
    val_start = train_end
    val_end = train_end + pd.DateOffset(months=VAL_MONTHS)

    windows.append({
        "train_start": train_start,
        "train_end": train_end,
        "val_start": val_start,
        "val_end": val_end,
    })

    current = current + pd.DateOffset(months=STRIDE_MONTHS)

print("Number of rolling windows:", len(windows))



Number of rolling windows: 20


In [5]:

# ============================================================
# 4. Load all window-based data and remove duplicate sequences
# ============================================================

all_datetime_sequences = []
all_latlon_sequences = []
metadata_rows = []

seen_sequence_ranges = set()

for window_index, w in enumerate(windows):

    bg_path = BG_WINDOW_DIR / f"Background_data-window_{window_index}.pkl"

    if not bg_path.exists():
        print(f"[SKIP] Missing file: {bg_path}")
        continue

    dfw = pd.read_pickle(bg_path)
    dfw = dfw.loc[:, ~dfw.columns.str.startswith("Q3")].copy()
    dfw.index = pd.to_datetime(dfw.index)

    split_data = {
        "train": dfw[(dfw.index >= w["train_start"]) & (dfw.index < w["train_end"])],
        "val": dfw[(dfw.index >= w["val_start"]) & (dfw.index < w["val_end"])],
    }

    for split_name, split_df in split_data.items():

        if len(split_df) == 0:
            continue

        dataset = HalfOrbitPairDataset(split_df, min_data_points=MIN_DATA_POINTS)
        created = dataset.create_half_orbit_sequences(split_df)

        datetime_sequences = created[1]
        latlon_sequences = created[2]

        added_count = 0

        for local_idx, (dt_seq, loc_seq) in enumerate(zip(datetime_sequences, latlon_sequences)):

            seq_start = pd.to_datetime(dt_seq[0])
            seq_end = pd.to_datetime(dt_seq[-1])

            duplicate_key = (seq_start, seq_end)

            if duplicate_key in seen_sequence_ranges:
                continue

            seen_sequence_ranges.add(duplicate_key)

            all_datetime_sequences.append(dt_seq)
            all_latlon_sequences.append(loc_seq)

            metadata_rows.append({
                "global_index": len(all_datetime_sequences) - 1,
                "window": window_index,
                "split": split_name,
                "local_sequence_index": local_idx,
                "sequence_start": seq_start,
                "sequence_end": seq_end,
            })

            added_count += 1

        print(
            f"Window {window_index:02d} | {split_name}: "
            f"kept {added_count} unique sequences"
        )

metadata_df = pd.DataFrame(metadata_rows)

metadata_path = RESULT_DIR / "unique_sequence_metadata_SW22.csv"
metadata_df.to_csv(metadata_path, index=False)

print("Total unique sequences:", len(all_datetime_sequences))
print("Saved metadata:", metadata_path)



[INFO] Total valid combined sequences: 1553
Window 00 | train: kept 1553 unique sequences
[INFO] Total valid combined sequences: 381
Window 00 | val: kept 381 unique sequences
[INFO] Total valid combined sequences: 1515
Window 01 | train: kept 1 unique sequences
[INFO] Total valid combined sequences: 434
Window 01 | val: kept 434 unique sequences
[INFO] Total valid combined sequences: 1572
Window 02 | train: kept 1 unique sequences
[INFO] Total valid combined sequences: 451
Window 02 | val: kept 451 unique sequences
[INFO] Total valid combined sequences: 1629
Window 03 | train: kept 469 unique sequences
[INFO] Total valid combined sequences: 408
Window 03 | val: kept 408 unique sequences
[INFO] Total valid combined sequences: 1694
Window 04 | train: kept 1 unique sequences
[INFO] Total valid combined sequences: 436
Window 04 | val: kept 436 unique sequences
[INFO] Total valid combined sequences: 1741
Window 05 | train: kept 429 unique sequences
[INFO] Total valid combined sequences: 49

In [6]:

# ============================================================
# 5. Random sampling baseline
# ============================================================

if len(all_datetime_sequences) < NUM_SAMPLES:
    raise ValueError(
        f"Only {len(all_datetime_sequences)} unique sequences available, "
        f"but NUM_SAMPLES={NUM_SAMPLES}."
    )

criteria_by_tw = {
    tw: SeismicCriteria(
        spatial_width=SPATIAL_WIDTH,
        time_window_hours=tw
    )
    for tw in TIME_WINDOWS
}

all_results = []
ratios_by_tw = {tw: [] for tw in TIME_WINDOWS}

for iteration in range(NUM_ITERATIONS):

    sampled_indices = rng.sample(
        range(len(all_datetime_sequences)),
        NUM_SAMPLES
    )

    # Same random offset is reused for all TWs in this iteration.
    # This makes TW comparison cleaner.
    sampled_anomaly_times = {}

    for seq_idx in sampled_indices:
        dt_seq = pd.to_datetime(all_datetime_sequences[seq_idx])
        random_days = rng.randint(
            RANDOM_OFFSET_MIN_DAYS,
            RANDOM_OFFSET_MAX_DAYS
        )
        sampled_anomaly_times[seq_idx] = dt_seq[-1] + pd.Timedelta(days=random_days)

    for tw in TIME_WINDOWS:

        criteria = criteria_by_tw[tw]
        seismic_count = 0

        for seq_idx in sampled_indices:

            anomaly_time = sampled_anomaly_times[seq_idx]
            loc_seq = all_latlon_sequences[seq_idx]

            is_sequence_seismic = 0

            for loc in loc_seq:
                is_seismic, inside_eqs, _ = criteria.is_eq(
                    anomaly_time,
                    loc,
                    eq
                )

                if is_seismic:
                    is_sequence_seismic = 1
                    break

            seismic_count += is_sequence_seismic

        seismic_ratio = seismic_count / NUM_SAMPLES
        ratios_by_tw[tw].append(seismic_ratio)

        all_results.append({
            "iteration": iteration + 1,
            "spatial_window": SPATIAL_WIDTH,
            "temporal_window": tw,
            "num_samples": NUM_SAMPLES,
            "seismic_count": seismic_count,
            "seismic_ratio": seismic_ratio,
            "random_offset_min_days": RANDOM_OFFSET_MIN_DAYS,
            "random_offset_max_days": RANDOM_OFFSET_MAX_DAYS,
        })

    if (iteration + 1) % 50 == 0:
        print(f"Completed {iteration + 1}/{NUM_ITERATIONS} iterations")



Completed 50/100 iterations
Completed 100/100 iterations


In [7]:

# ============================================================
# 6. Save full 1000-sampling results
# ============================================================

results_df = pd.DataFrame(all_results)

csv_path = RESULT_DIR / (
    f"RST_simple_1000samples_200seq_SW{SPATIAL_WIDTH}_"
    f"TW{'-'.join(map(str, TIME_WINDOWS))}.csv"
)

pkl_path = RESULT_DIR / (
    f"RST_simple_1000samples_200seq_SW{SPATIAL_WIDTH}_"
    f"TW{'-'.join(map(str, TIME_WINDOWS))}.pkl"
)

results_df.to_csv(csv_path, index=False)

with open(pkl_path, "wb") as f:
    pickle.dump(
        {
            "ratios_by_tw": ratios_by_tw,
            "full_results": results_df,
            "metadata": {
                "spatial_window": SPATIAL_WIDTH,
                "temporal_windows": TIME_WINDOWS,
                "num_samples": NUM_SAMPLES,
                "num_iterations": NUM_ITERATIONS,
                "random_offset_min_days": RANDOM_OFFSET_MIN_DAYS,
                "random_offset_max_days": RANDOM_OFFSET_MAX_DAYS,
                "seed": SEED,
            },
        },
        f
    )

print("Saved full CSV:", csv_path)
print("Saved pickle:", pkl_path)


Saved full CSV: C:\PROJECT-DEMETER\DICLUB\GRID-METHODS\LSTM-AE-HO\Demeter\Final\Results\Random_baseline_simple\RST_simple_1000samples_200seq_SW22_TW6-12-24-48.csv
Saved pickle: C:\PROJECT-DEMETER\DICLUB\GRID-METHODS\LSTM-AE-HO\Demeter\Final\Results\Random_baseline_simple\RST_simple_1000samples_200seq_SW22_TW6-12-24-48.pkl


In [8]:



summary_rows = []

for tw, ratios in ratios_by_tw.items():
    ratios = np.asarray(ratios, dtype=float)

    summary_rows.append({
        "spatial_window": SPATIAL_WIDTH,
        "temporal_window": tw,
        "mean": np.mean(ratios),
        "median": np.median(ratios),
        "std": np.std(ratios),
        "p0_3": np.percentile(ratios, 0.3),
        "p99_7": np.percentile(ratios, 99.7),
        "num_iterations": NUM_ITERATIONS,
        "num_samples": NUM_SAMPLES,
    })

summary_df = pd.DataFrame(summary_rows)

summary_path = RESULT_DIR / (
    f"RST_simple_summary_SW{SPATIAL_WIDTH}_"
    f"TW{'-'.join(map(str, TIME_WINDOWS))}.csv"
)

summary_df.to_csv(summary_path, index=False)

print("Saved summary CSV:", summary_path)
print(summary_df)

Saved summary CSV: C:\PROJECT-DEMETER\DICLUB\GRID-METHODS\LSTM-AE-HO\Demeter\Final\Results\Random_baseline_simple\RST_simple_summary_SW22_TW6-12-24-48.csv
   spatial_window  temporal_window    mean  median       std     p0_3  \
0              22                6  0.1390    0.15  0.073682  0.00000   
1              22               12  0.2355    0.20  0.099824  0.01485   
2              22               24  0.3765    0.35  0.108732  0.15000   
3              22               48  0.5605    0.55  0.120061  0.30000   

     p99_7  num_iterations  num_samples  
0  0.33515             100           20  
1  0.48515             100           20  
2  0.60000             100           20  
3  0.87030             100           20  
